# CommissionLens — Jupyter Notebook
**Commission-Adjusted Alpha Prediction in Indian Mutual Funds**

FEC · IIT Guwahati | Mentor: Harshita Sharma

---
This notebook walks through the entire CommissionLens pipeline:
1. Data collection from AMFI, NSE, and RBI sources
2. Feature engineering — expense gap, rolling Sharpe, beta, IR
3. ML model training (XGBoost) with temporal train/test split
4. SHAP explainability — top predictors of commission justification
5. SIP back-validation — XIRR comparison (model vs naive)

In [ ]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Dark theme
plt.rcParams.update({
    'figure.facecolor': '#1e1e1e', 'axes.facecolor': '#1e1e1e',
    'axes.labelcolor': '#ccc', 'xtick.color': '#ccc', 'ytick.color': '#ccc',
    'text.color': '#ccc', 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#333', 'grid.color': '#333',
})
print('✓ Imports ready')

## 1. Data Collection

In [ ]:
from data_fetcher import (
    fetch_amfi_fund_list, filter_equity_funds, build_fund_pairs,
    fetch_nifty50_from_nse, fetch_rbi_macro_data, generate_fii_dii_flows
)

# Fetch fund list
all_funds = fetch_amfi_fund_list()
print(f'Total schemes from AMFI: {len(all_funds)}')
all_funds.head(3)

In [ ]:
equity = filter_equity_funds(all_funds)
pairs = build_fund_pairs(equity)
print(f'Direct/Regular pairs matched: {len(pairs)}')
pairs.head(5)

In [ ]:
# Fetch Nifty50 and macro
nifty = fetch_nifty50_from_nse()
macro = fetch_rbi_macro_data()
fii_dii = generate_fii_dii_flows()
macro = macro.merge(fii_dii, on='date', how='left')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(nifty['date'], nifty['nifty50_close'], color='#4a9eff', linewidth=1.2)
axes[0].set_title('Nifty 50', color='#f5a623')
axes[0].set_ylabel('Index Level')

axes[1].plot(macro['date'], macro['repo_rate'], color='#f5a623', label='Repo Rate')
axes[1].plot(macro['date'], macro['cpi_yoy'], color='#f44336', label='CPI YoY')
axes[1].set_title('Macro Indicators', color='#f5a623')
axes[1].legend(facecolor='#1e1e1e', edgecolor='#333', labelcolor='white')

plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('../data')

# Load pre-fetched NAV data (run pipeline.py first)
nav_df = pd.read_parquet(DATA_DIR / 'nav_history.parquet')
print(f'NAV rows: {len(nav_df)} | Funds: {nav_df["scheme_code"].nunique()}')
nav_df.head(3)

In [ ]:
features = pd.read_parquet(DATA_DIR / 'features.parquet')
features['quarter_end'] = pd.to_datetime(features['quarter_end'])
print(f'Feature rows: {len(features)}')
print(f'Commission justified: {features["label_commission_justified"].mean():.1%}')
features.describe().round(4)

In [ ]:
# Expense gap distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(features['expense_gap_annualized'].dropna() * 100, bins=30,
             color='#f5a623', edgecolor='none', alpha=0.85)
axes[0].set_title('Expense Gap Distribution')
axes[0].set_xlabel('Expense Gap (% annualized)')

axes[1].hist(features['net_alpha'].dropna() * 100, bins=30,
             color='#4a9eff', edgecolor='none', alpha=0.85)
axes[1].axvline(0, color='white', linestyle='--')
axes[1].set_title('Net Alpha Distribution')
axes[1].set_xlabel('Net Alpha (%)')

axes[2].hist(features['sharpe_ratio_1y'].dropna(), bins=30,
             color='#4caf50', edgecolor='none', alpha=0.85)
axes[2].set_title('1Y Sharpe Ratio')
axes[2].set_xlabel('Sharpe Ratio')

plt.tight_layout()
plt.show()

## 3. Model Training

In [ ]:
from model_training import (
    load_features, temporal_train_test_split, prepare_arrays,
    train_classifier, train_regressor,
    evaluate_classifier, evaluate_regressor
)

df = load_features()
train, val, test = temporal_train_test_split(df)

X_train, y_cls_train, y_reg_train, feat_names = prepare_arrays(train)
X_val, y_cls_val, y_reg_val, _ = prepare_arrays(val)
X_test, y_cls_test, y_reg_test, _ = prepare_arrays(test)

print(f'Features: {feat_names}')

In [ ]:
print('Training classifier...')
clf = train_classifier(X_train, y_cls_train, X_val, y_cls_val, feat_names)

print('\nEvaluating...')
cls_metrics = evaluate_classifier(clf, X_test, y_cls_test, feat_names)

In [ ]:
print('Training regressor...')
reg_model = train_regressor(X_train, y_reg_train, X_val, y_reg_val, feat_names)
reg_metrics = evaluate_regressor(reg_model, X_test, y_reg_test)

## 4. SHAP Explainability

In [ ]:
import shap

explainer = shap.TreeExplainer(clf)
shap_vals = explainer.shap_values(X_test)

if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]  # positive class

import pandas as pd
shap_df = pd.DataFrame(shap_vals, columns=feat_names)
mean_abs = shap_df.abs().mean().sort_values(ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#4a9eff' if v > 0 else '#f44336'
          for v in shap_df[mean_abs.index].mean()]
ax.barh(mean_abs.index, mean_abs.values, color=colors[::-1], edgecolor='none', alpha=0.85)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('SHAP Feature Importance — Commission Justification', color='#f5a623')
plt.tight_layout()
plt.show()

In [ ]:
# Waterfall plot for a single fund
sample_idx = 0
shap.waterfall_plot(
    shap.Explanation(
        values=shap_vals[sample_idx],
        base_values=explainer.expected_value if not isinstance(explainer.expected_value, list)
                    else explainer.expected_value[1],
        data=X_test[sample_idx],
        feature_names=feat_names
    )
)

## 5. SIP Back-Validation

In [ ]:
import json
from pathlib import Path

report_path = Path('../reports/model_report.json')
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    bt = report.get('sip_backtest', {})
    if bt:
        print(f"Total Invested:         ₹{bt['total_invested']:,.0f}")
        print(f"Model Portfolio Value:  ₹{bt['model_final_value']:,.0f} (XIRR: {bt['model_xirr']}%)")
        print(f"Naive Regular Value:    ₹{bt['naive_final_value']:,.0f} (XIRR: {bt['naive_xirr']}%)")
        print(f"Excess Return:          {bt['excess_return_pct']}% p.a.")

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(['Naive Regular', 'Model-Guided Direct'],
               [bt['naive_final_value'], bt['model_final_value']],
               color=['#f44336', '#4caf50'], width=0.45)
        ax.set_ylabel('Final Portfolio Value (₹)')
        ax.set_title(f'SIP ₹5,000/month — 2019-2023', color='#f5a623')
        plt.tight_layout()
        plt.show()
    else:
        print('SIP backtest data not found in report.')
else:
    print('Run pipeline.py first.')

## 6. Concepts Mastered

| Concept | Implementation |
|---|---|
| **Net Alpha** = Gross Return − Benchmark − Expense Gap | `feature_engineering.py::compute_expense_gap` |
| **Information Ratio** & alpha persistence | `feature_engineering.py::compute_rolling_features` |
| **XIRR** for SIP cash flows | `model_training.py::xirr_newton` |
| **Temporal train-test split** (no look-ahead bias) | `model_training.py::temporal_train_test_split` |
| **SHAP feature importance** | `model_training.py::compute_shap_values` |
| **Rupee Cost Averaging** & market regimes | Macro features: repo_rate, CPI, yield_spread |
| **Portfolio Turnover** & hidden trading costs | `feature_engineering.py::estimate_turnover` |